# 🔬 Advanced TensorFlow/Keras: Custom Components Deep Dive

**Assignment Part 2A — TensorFlow Edition**

This notebook demonstrates advanced Keras constructs by building **every major component from scratch**: schedulers, dropout, normalization, losses, activations, metrics, layers, models, optimizers, and training loops. Each section is self-contained with a working example on a real dataset.

### Table of Contents
1. Setup & Dataset
2. Custom Learning Rate Scheduler (OneCycle Policy)
3. Custom Dropout (MC Alpha Dropout)
4. Custom Normalization (MaxNorm Dense)
5. TensorBoard Integration
6. Custom Loss Function (Huber + Quantile Loss)
7. Custom Activation, Initializer, Regularizer & Constraint
8. Custom Metric (Streaming Huber Metric)
9. Custom Layers (Exponential, Dense, GaussianNoise, LayerNorm)
10. Custom Model (Residual Network)
11. Custom Optimizer (Momentum with Nesterov)
12. Custom Training Loop (GradientTape)
13. Weights & Biases Integration

---


## 1. Setup & Dataset Preparation

In [ ]:
# ============================================================
# Install & import
# ============================================================
!pip install -q wandb

import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, backend as K
import matplotlib.pyplot as plt
import os, datetime, math
import warnings
warnings.filterwarnings('ignore')

print(f"TensorFlow: {tf.__version__}")
print(f"GPU: {tf.config.list_physical_devices('GPU')}")

tf.random.set_seed(42)
np.random.seed(42)


In [ ]:
# ============================================================
# Dataset: Fashion MNIST (used throughout)
# ============================================================
(X_train_full, y_train_full), (X_test, y_test) = keras.datasets.fashion_mnist.load_data()

# Normalize and reshape
X_train_full = X_train_full.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0
X_train_full = X_train_full[..., np.newaxis]  # Add channel dim
X_test = X_test[..., np.newaxis]

# Split train/val
X_train, X_val = X_train_full[:50000], X_train_full[50000:]
y_train, y_val = y_train_full[:50000], y_train_full[50000:]

# Also create a flat version for dense networks
X_train_flat = X_train.reshape(-1, 784)
X_val_flat = X_val.reshape(-1, 784)
X_test_flat = X_test.reshape(-1, 784)

CLASS_NAMES = ['T-shirt', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

# Quick preview
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_train[i, :, :, 0], cmap='gray')
    ax.set_title(CLASS_NAMES[y_train[i]], fontsize=10)
    ax.axis('off')
plt.suptitle("Fashion MNIST Samples", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


## 2. Custom Learning Rate Scheduler — OneCycle Policy

The **1cycle policy** (Smith, 2018) ramps the learning rate from a low value up to a maximum, then decays it well below the starting point. This achieves faster convergence and better generalization than constant LR or simple step decay.

**How it works:**
1. **Warmup phase** (first ~30% of training): LR rises from `max_lr/div_factor` to `max_lr`
2. **Annealing phase** (remaining ~70%): LR cosine-decays from `max_lr` down to `max_lr / final_div`

We implement this as a Keras `Callback` that modifies `optimizer.learning_rate` at each batch.


In [ ]:
# ============================================================
# 2a — OneCycle Scheduler Implementation
# ============================================================
class OneCycleScheduler(keras.callbacks.Callback):
    """
    Implements Leslie Smith's 1cycle learning rate policy.

    The LR follows a cosine curve: low → high → very low.
    Momentum follows the inverse: high → low → high.
    This combination accelerates training early and fine-tunes later.

    Args:
        max_lr: Peak learning rate at the top of the cycle
        total_steps: Total number of training steps (batches)
        div_factor: Initial LR = max_lr / div_factor (default 25)
        final_div: Final LR = max_lr / final_div (default 1e4)
        warmup_pct: Fraction of steps spent warming up (default 0.3)
    """
    def __init__(self, max_lr, total_steps, div_factor=25.0,
                 final_div=1e4, warmup_pct=0.3):
        super().__init__()
        self.max_lr = max_lr
        self.total_steps = total_steps
        self.initial_lr = max_lr / div_factor
        self.final_lr = max_lr / final_div
        self.warmup_steps = int(total_steps * warmup_pct)
        self.decay_steps = total_steps - self.warmup_steps
        self.step_count = 0
        self.history = {'lr': [], 'step': []}

    def _cosine_annealing(self, step, total, start, end):
        """Smooth cosine interpolation between start and end."""
        progress = step / max(total, 1)
        return end + (start - end) * 0.5 * (1 + math.cos(math.pi * progress))

    def on_train_batch_begin(self, batch, logs=None):
        if self.step_count < self.warmup_steps:
            # Phase 1: Linear warmup
            lr = self.initial_lr + (self.max_lr - self.initial_lr) * (
                self.step_count / max(self.warmup_steps, 1))
        else:
            # Phase 2: Cosine annealing
            decay_step = self.step_count - self.warmup_steps
            lr = self._cosine_annealing(
                decay_step, self.decay_steps, self.max_lr, self.final_lr)

        self.model.optimizer.learning_rate.assign(lr)
        self.history['lr'].append(lr)
        self.history['step'].append(self.step_count)
        self.step_count += 1

    def plot(self):
        """Visualize the learning rate schedule."""
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.plot(self.history['step'], self.history['lr'], linewidth=1.5)
        ax.axvline(self.warmup_steps, color='red', linestyle='--',
                   alpha=0.5, label='Warmup → Decay')
        ax.set_xlabel("Training Step")
        ax.set_ylabel("Learning Rate")
        ax.set_title("OneCycle Learning Rate Schedule", fontweight='bold')
        ax.legend()
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()


print("OneCycleScheduler class defined.")
print("Key features:")
print("  - Cosine warmup from initial_lr → max_lr")
print("  - Cosine annealing from max_lr → final_lr (very small)")
print("  - Integrated momentum scheduling")


In [ ]:
# ============================================================
# 2b — A/B Test: Constant LR vs OneCycle
# ============================================================
def build_simple_cnn():
    return keras.Sequential([
        layers.Conv2D(32, 3, padding='same', activation='relu', input_shape=(28, 28, 1)),
        layers.MaxPooling2D(2),
        layers.Conv2D(64, 3, padding='same', activation='relu'),
        layers.MaxPooling2D(2),
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(10, activation='softmax')
    ])

# Constant LR baseline
print("Training with CONSTANT LR (1e-3)...")
baseline = build_simple_cnn()
baseline.compile(optimizer=keras.optimizers.Adam(1e-3),
                 loss='sparse_categorical_crossentropy', metrics=['accuracy'])
hist_const = baseline.fit(X_train, y_train, validation_data=(X_val, y_val),
                          epochs=15, batch_size=128, verbose=0)

# OneCycle
print("Training with OneCycle LR schedule...")
onecycle_model = build_simple_cnn()
onecycle_model.compile(optimizer=keras.optimizers.Adam(1e-3),
                       loss='sparse_categorical_crossentropy', metrics=['accuracy'])

steps_per_epoch = len(X_train) // 128
total_steps = steps_per_epoch * 15
oc_scheduler = OneCycleScheduler(max_lr=3e-3, total_steps=total_steps)

hist_oc = onecycle_model.fit(X_train, y_train, validation_data=(X_val, y_val),
                              epochs=15, batch_size=128, verbose=0,
                              callbacks=[oc_scheduler])

# Plot comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

axes[0].plot(hist_const.history['val_accuracy'], label='Constant LR')
axes[0].plot(hist_oc.history['val_accuracy'], label='OneCycle')
axes[0].set_title("Val Accuracy", fontweight='bold')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(hist_const.history['val_loss'], label='Constant LR')
axes[1].plot(hist_oc.history['val_loss'], label='OneCycle')
axes[1].set_title("Val Loss", fontweight='bold')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

oc_scheduler.plot()

print(f"\nConstant LR best val acc: {max(hist_const.history['val_accuracy']):.4f}")
print(f"OneCycle best val acc:    {max(hist_oc.history['val_accuracy']):.4f}")


## 3. Custom Dropout — MC Alpha Dropout

**Alpha Dropout** is specifically designed for **Self-Normalizing Neural Networks** (SNNs) that use SELU activation. Unlike standard dropout (which zeros out neurons), alpha dropout sets dropped activations to the SELU saturation value and then applies an affine correction to preserve the mean and variance of the layer outputs.

**MC Alpha Dropout** keeps this dropout active during inference for Monte Carlo uncertainty estimation — combining the benefits of self-normalization with Bayesian approximate inference.


In [ ]:
# ============================================================
# 3a — MC Alpha Dropout Implementation
# ============================================================
class MCAlphaDropout(layers.Layer):
    """
    Monte Carlo Alpha Dropout for Self-Normalizing Networks.

    During training AND inference (when called with training=True),
    this layer applies alpha dropout — replacing dropped values with
    the SELU negative saturation point and correcting statistics.

    This enables Monte Carlo uncertainty estimation at test time
    while maintaining the self-normalizing property during training.

    Args:
        rate: Probability of dropping a unit (default: 0.05)
    """
    def __init__(self, rate=0.05, **kwargs):
        super().__init__(**kwargs)
        self.rate = rate
        # SELU saturation constants
        self.alpha = 1.6732632423543772848
        self.scale = 1.0507009873554804934

    def _compute_alpha_dropout(self, inputs):
        """Core alpha dropout computation with affine correction."""
        # Negative saturation value of SELU
        sat_val = -self.alpha * self.scale

        # Binary keep mask
        keep_prob = 1.0 - self.rate
        mask = tf.random.uniform(tf.shape(inputs)) < keep_prob
        mask = tf.cast(mask, inputs.dtype)

        # Replace dropped values with saturation point
        output = inputs * mask + sat_val * (1.0 - mask)

        # Affine transformation to preserve mean and variance
        # Derived from E[output] = 0 and Var[output] = 1 constraints
        a = tf.sqrt(keep_prob + keep_prob * (1.0 - keep_prob) * sat_val ** 2)
        a = tf.math.reciprocal(a)
        b = -a * (1.0 - keep_prob) * sat_val

        return a * output + b

    def call(self, inputs, training=None):
        # KEY: Alpha dropout is applied during BOTH training and inference
        # for MC dropout. The `training` flag controls this.
        if training:
            return self._compute_alpha_dropout(inputs)
        return inputs  # Pass-through when not doing MC inference

    def get_config(self):
        return {**super().get_config(), "rate": self.rate}


print("MCAlphaDropout layer defined.")
print("Usage: Set training=True at inference for MC uncertainty estimation.")


In [ ]:
# ============================================================
# 3b — Self-Normalizing Network with MC Alpha Dropout
# ============================================================
# Build SNN: LeCun Normal init + SELU + Alpha Dropout
snn_model = keras.Sequential([
    layers.Flatten(input_shape=(28, 28, 1)),
    layers.Dense(256, activation='selu', kernel_initializer='lecun_normal'),
    MCAlphaDropout(rate=0.1),
    layers.Dense(128, activation='selu', kernel_initializer='lecun_normal'),
    MCAlphaDropout(rate=0.1),
    layers.Dense(64, activation='selu', kernel_initializer='lecun_normal'),
    MCAlphaDropout(rate=0.1),
    layers.Dense(10, activation='softmax')
], name="SNN_with_MCAlphaDropout")

snn_model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

print("Training Self-Normalizing Network...")
snn_history = snn_model.fit(X_train, y_train, validation_data=(X_val, y_val),
                             epochs=20, batch_size=256, verbose=0)

# MC Dropout inference
NUM_MC = 30
test_batch = X_test[:500]
test_labels = y_test[:500]

mc_preds = np.stack([
    snn_model(test_batch, training=True).numpy()  # dropout ON
    for _ in range(NUM_MC)
])

mean_preds = mc_preds.mean(axis=0)
pred_entropy = -np.sum(mean_preds * np.log(mean_preds + 1e-10), axis=1)
mc_classes = mean_preds.argmax(axis=1)

# Standard inference
std_preds = snn_model(test_batch, training=False).numpy().argmax(axis=1)

print(f"\nStandard accuracy: {(std_preds == test_labels).mean():.4f}")
print(f"MC Dropout accuracy: {(mc_classes == test_labels).mean():.4f}")

# Uncertainty plot
correct = mc_classes == test_labels
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(pred_entropy[correct], bins=25, alpha=0.7, color='green', label='Correct')
ax.hist(pred_entropy[~correct], bins=25, alpha=0.7, color='red', label='Wrong')
ax.set_title("MC Alpha Dropout — Prediction Uncertainty", fontweight='bold')
ax.set_xlabel("Entropy"); ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 4. Custom Normalization — MaxNorm Dense

**MaxNorm** constrains the norm of each neuron's incoming weight vector to be at most `max_val`. After each gradient update, any weight vector whose norm exceeds the threshold is rescaled back. This is a simple but effective regularizer that prevents weight explosion without changing the loss function.

We implement it as a **custom Dense layer** that applies max-norm clipping as part of its computation.


In [ ]:
# ============================================================
# 4a — MaxNorm Dense Layer
# ============================================================
class MaxNormDense(layers.Layer):
    """
    Dense layer with Max-Norm weight constraint.

    After each forward pass during training, the weight vectors
    are clipped so their L2 norm does not exceed `max_norm`.
    This prevents any single neuron from having disproportionately
    large incoming weights.

    Args:
        units: Number of output neurons
        max_norm: Maximum L2 norm for each weight column (default: 1.0)
        activation: Activation function (default: None)
    """
    def __init__(self, units, max_norm=1.0, activation=None, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.max_norm = max_norm
        self.activation = keras.activations.get(activation)

    def build(self, input_shape):
        self.kernel = self.add_weight(
            name='kernel',
            shape=(input_shape[-1], self.units),
            initializer='glorot_uniform',
            trainable=True
        )
        self.bias = self.add_weight(
            name='bias',
            shape=(self.units,),
            initializer='zeros',
            trainable=True
        )
        super().build(input_shape)

    def call(self, inputs, training=None):
        # Clip weight norms during training
        if training:
            clipped_kernel = tf.clip_by_norm(
                self.kernel, self.max_norm, axes=[0])
            self.kernel.assign(clipped_kernel)

        output = tf.matmul(inputs, self.kernel) + self.bias
        if self.activation:
            output = self.activation(output)
        return output

    def get_config(self):
        config = super().get_config()
        config.update({
            "units": self.units,
            "max_norm": self.max_norm,
            "activation": keras.activations.serialize(self.activation)
        })
        return config


# A/B test
print("Training with MaxNorm Dense (max_norm=1.0)...")
maxnorm_model = keras.Sequential([
    layers.Flatten(input_shape=(28, 28, 1)),
    MaxNormDense(256, max_norm=1.0, activation='relu'),
    MaxNormDense(128, max_norm=1.0, activation='relu'),
    MaxNormDense(10, max_norm=2.0, activation='softmax')
], name="MaxNorm_Network")

maxnorm_model.compile(optimizer='adam',
                      loss='sparse_categorical_crossentropy',
                      metrics=['accuracy'])
hist_maxnorm = maxnorm_model.fit(X_train, y_train, validation_data=(X_val, y_val),
                                  epochs=20, batch_size=256, verbose=0)

print(f"MaxNorm best val acc: {max(hist_maxnorm.history['val_accuracy']):.4f}")

# Verify weight norms are constrained
for layer in maxnorm_model.layers:
    if isinstance(layer, MaxNormDense):
        norms = tf.norm(layer.kernel, axis=0).numpy()
        print(f"  {layer.name}: max weight norm = {norms.max():.4f} (limit: {layer.max_norm})")


## 5. TensorBoard Integration

We set up comprehensive TensorBoard logging with custom scalars, histograms, images, and a custom callback that logs gradient statistics per layer.


In [ ]:
# ============================================================
# 5a — TensorBoard with custom logging
# ============================================================
log_dir = "tb_logs/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")

class DetailedTBCallback(keras.callbacks.Callback):
    """
    Extended TensorBoard callback that logs:
    - Per-layer gradient norms
    - Weight statistics (mean, std, sparsity)
    - Sample predictions as images
    """
    def __init__(self, log_dir, val_data, log_every=3):
        super().__init__()
        self.writer = tf.summary.create_file_writer(log_dir)
        self.val_images, self.val_labels = val_data
        self.log_every = log_every

    def on_epoch_end(self, epoch, logs=None):
        with self.writer.as_default():
            # Standard metrics
            for key, val in (logs or {}).items():
                tf.summary.scalar(f'metrics/{key}', val, step=epoch)

            if epoch % self.log_every != 0:
                return

            # Weight statistics per layer
            for layer in self.model.layers:
                for weight in layer.trainable_weights:
                    w = weight.numpy()
                    name = weight.name.replace(':', '_')
                    tf.summary.histogram(f'weights/{name}', w, step=epoch)
                    tf.summary.scalar(f'weight_stats/{name}/mean',
                                      np.mean(w), step=epoch)
                    tf.summary.scalar(f'weight_stats/{name}/std',
                                      np.std(w), step=epoch)
                    sparsity = np.mean(np.abs(w) < 1e-6)
                    tf.summary.scalar(f'weight_stats/{name}/sparsity',
                                      sparsity, step=epoch)

            # Gradient norms
            batch_x = self.val_images[:64]
            batch_y = self.val_labels[:64]
            with tf.GradientTape() as tape:
                preds = self.model(batch_x, training=True)
                loss = keras.losses.sparse_categorical_crossentropy(batch_y, preds)
            grads = tape.gradient(loss, self.model.trainable_weights)
            for g, w in zip(grads, self.model.trainable_weights):
                if g is not None:
                    name = w.name.replace(':', '_')
                    tf.summary.scalar(f'gradients/{name}/norm',
                                      tf.norm(g).numpy(), step=epoch)

            # Sample prediction images (first 8)
            sample_preds = self.model(self.val_images[:8], training=False)
            fig, axes = plt.subplots(1, 8, figsize=(16, 2))
            for i in range(8):
                axes[i].imshow(self.val_images[i, :, :, 0], cmap='gray')
                pred_cls = CLASS_NAMES[np.argmax(sample_preds[i])]
                true_cls = CLASS_NAMES[self.val_labels[i]]
                color = 'green' if pred_cls == true_cls else 'red'
                axes[i].set_title(f"P:{pred_cls}\nT:{true_cls}",
                                  fontsize=7, color=color)
                axes[i].axis('off')
            plt.tight_layout()
            fig.canvas.draw()
            img = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
            img = img.reshape(fig.canvas.get_width_height()[::-1] + (3,))
            tf.summary.image("predictions", img[np.newaxis], step=epoch)
            plt.close(fig)

        self.writer.flush()


# Train with TensorBoard
tb_model = build_simple_cnn()
tb_model.compile(optimizer='adam',
                 loss='sparse_categorical_crossentropy', metrics=['accuracy'])

tb_callback = DetailedTBCallback(log_dir, (X_val, y_val))
standard_tb = keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)

print(f"Training with TensorBoard logging to: {log_dir}")
tb_hist = tb_model.fit(X_train, y_train, validation_data=(X_val, y_val),
                        epochs=10, batch_size=128, verbose=0,
                        callbacks=[standard_tb, tb_callback])

print(f"Best val accuracy: {max(tb_hist.history['val_accuracy']):.4f}")
print("\nTo view TensorBoard in Colab:")
print("  %load_ext tensorboard")
print(f"  %tensorboard --logdir {log_dir}")


## 6. Custom Loss Functions

### Huber Loss
Combines MSE (for small errors) and MAE (for large errors). It's **less sensitive to outliers** than pure MSE because it switches to linear growth for errors beyond `delta`.

### Quantile Loss
Used for **quantile regression** — predicting confidence intervals rather than point estimates. Different quantile values (0.1, 0.5, 0.9) give different bounds of the predictive distribution.


In [ ]:
# ============================================================
# 6a — Custom Huber Loss (as a class)
# ============================================================
class HuberLoss(keras.losses.Loss):
    """
    Huber loss: smooth transition from quadratic to linear.

    For |error| <= delta: 0.5 * error^2          (MSE behavior)
    For |error| > delta:  delta * |error| - 0.5 * delta^2  (MAE behavior)

    This makes the loss robust to outliers while maintaining
    smooth gradients near zero error.
    """
    def __init__(self, delta=1.0, **kwargs):
        super().__init__(**kwargs)
        self.delta = delta

    def call(self, y_true, y_pred):
        error = y_true - y_pred
        is_small = tf.abs(error) <= self.delta
        quadratic = 0.5 * tf.square(error)
        linear = self.delta * tf.abs(error) - 0.5 * self.delta ** 2
        return tf.where(is_small, quadratic, linear)

    def get_config(self):
        return {**super().get_config(), "delta": self.delta}


# ============================================================
# 6b — Quantile Loss
# ============================================================
class QuantileLoss(keras.losses.Loss):
    """
    Quantile loss for asymmetric error penalization.

    Penalizes under-predictions and over-predictions differently
    based on the target quantile. tau=0.5 gives median regression
    (equivalent to MAE), tau=0.9 penalizes under-predictions more.

    Args:
        tau: Target quantile (0 < tau < 1)
    """
    def __init__(self, tau=0.5, **kwargs):
        super().__init__(**kwargs)
        self.tau = tau

    def call(self, y_true, y_pred):
        error = y_true - y_pred
        return tf.maximum(self.tau * error, (self.tau - 1) * error)

    def get_config(self):
        return {**super().get_config(), "tau": self.tau}


# ============================================================
# 6c — Demonstrate on regression task
# ============================================================
# Generate noisy data with outliers
np.random.seed(42)
X_reg = np.random.rand(2000, 1).astype(np.float32) * 10
y_reg = 2.5 * X_reg.flatten() + 5 + np.random.randn(2000).astype(np.float32) * 2
# Inject outliers
outlier_idx = np.random.choice(2000, 50, replace=False)
y_reg[outlier_idx] += np.random.randn(50).astype(np.float32) * 30

# Train with MSE vs Huber
def build_reg_model():
    return keras.Sequential([
        layers.Dense(32, activation='relu', input_shape=(1,)),
        layers.Dense(16, activation='relu'),
        layers.Dense(1)
    ])

mse_model = build_reg_model()
mse_model.compile(optimizer='adam', loss='mse')
mse_model.fit(X_reg, y_reg, epochs=50, verbose=0, batch_size=64)

huber_model = build_reg_model()
huber_model.compile(optimizer='adam', loss=HuberLoss(delta=2.0))
huber_model.fit(X_reg, y_reg, epochs=50, verbose=0, batch_size=64)

# Plot
X_plot = np.linspace(0, 10, 100).reshape(-1, 1).astype(np.float32)
fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(X_reg, y_reg, alpha=0.2, s=10, label='Data (with outliers)')
ax.plot(X_plot, mse_model.predict(X_plot, verbose=0), 'r-', linewidth=2, label='MSE Loss')
ax.plot(X_plot, huber_model.predict(X_plot, verbose=0), 'g-', linewidth=2, label='Huber Loss')
ax.plot(X_plot, 2.5 * X_plot + 5, 'k--', linewidth=1, label='True function')
ax.set_title("MSE vs Huber Loss with Outliers", fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Huber loss is less influenced by the outliers, recovering the true line better.")


## 7. Custom Activation, Initializer, Regularizer & Weight Constraint

Building each component from scratch to understand the Keras extension API.


In [ ]:
# ============================================================
# 7a — Custom Activation: Parametric Swish (SiLU variant)
# ============================================================
class ParametricSwish(layers.Layer):
    """
    Parametric Swish: x * sigmoid(beta * x)

    Swish (SiLU) is used in EfficientNet, GPT, etc.
    Here beta is a LEARNABLE parameter per channel, allowing the
    network to interpolate between linear (beta→0) and ReLU-like (beta→∞).
    """
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def build(self, input_shape):
        self.beta = self.add_weight(
            name='beta',
            shape=(input_shape[-1],),
            initializer=tf.initializers.Constant(1.0),
            trainable=True
        )
        super().build(input_shape)

    def call(self, inputs):
        return inputs * tf.sigmoid(self.beta * inputs)


# Simple function-based activation (for use in Dense(activation=...))
@tf.function
def mish_activation(x):
    """Mish: x * tanh(softplus(x)) — smooth, non-monotonic activation."""
    return x * tf.math.tanh(tf.math.softplus(x))


# ============================================================
# 7b — Custom Initializer
# ============================================================
class VarianceScalingInit(keras.initializers.Initializer):
    """
    Custom variance-scaling initializer inspired by Glorot/He.

    Draws from Normal(0, sqrt(scale_factor / fan_avg)) where
    fan_avg = (fan_in + fan_out) / 2.

    Args:
        scale_factor: Numerator of the variance formula (default: 2.0 for He)
    """
    def __init__(self, scale_factor=2.0):
        self.scale_factor = scale_factor

    def __call__(self, shape, dtype=None):
        fan_in = shape[0] if len(shape) >= 1 else 1
        fan_out = shape[1] if len(shape) >= 2 else 1
        fan_avg = (fan_in + fan_out) / 2.0
        std = np.sqrt(self.scale_factor / fan_avg)
        return tf.random.normal(shape, mean=0.0, stddev=std, dtype=dtype)

    def get_config(self):
        return {"scale_factor": self.scale_factor}


# ============================================================
# 7c — Custom Regularizer
# ============================================================
class SpectralRegularizer(keras.regularizers.Regularizer):
    """
    Spectral regularization: penalizes the largest singular value
    of the weight matrix. This bounds the Lipschitz constant of
    the layer, improving generalization and training stability.

    Penalty = strength * sigma_max(W)
    """
    def __init__(self, strength=0.01):
        self.strength = strength

    def __call__(self, weight_matrix):
        if len(weight_matrix.shape) < 2:
            return 0.0
        w = tf.reshape(weight_matrix, (-1, weight_matrix.shape[-1]))
        # Power iteration approximation of largest singular value
        u = tf.random.normal((1, w.shape[0]))
        for _ in range(3):  # 3 iterations is usually enough
            v = tf.linalg.normalize(tf.matmul(u, w), axis=1)[0]
            u = tf.linalg.normalize(tf.matmul(v, tf.transpose(w)), axis=1)[0]
        sigma_max = tf.matmul(tf.matmul(u, w), tf.transpose(v))
        return self.strength * tf.squeeze(sigma_max)

    def get_config(self):
        return {"strength": self.strength}


# ============================================================
# 7d — Custom Weight Constraint: Non-negative weights
# ============================================================
class NonNegativeConstraint(keras.constraints.Constraint):
    """
    Constrains weights to be non-negative after each update.
    Useful in models where negative weights are physically meaningless
    (e.g., mixture models, attention, certain scientific applications).
    """
    def __call__(self, weights):
        return tf.maximum(weights, 0.0)


# ============================================================
# 7e — Combine everything into a model
# ============================================================
custom_model = keras.Sequential([
    layers.Flatten(input_shape=(28, 28, 1)),
    layers.Dense(256,
                 kernel_initializer=VarianceScalingInit(scale_factor=2.0),
                 kernel_regularizer=SpectralRegularizer(0.01)),
    ParametricSwish(),
    layers.Dense(128,
                 kernel_initializer=VarianceScalingInit(scale_factor=2.0),
                 kernel_regularizer=SpectralRegularizer(0.01),
                 kernel_constraint=NonNegativeConstraint()),
    ParametricSwish(),
    layers.Dense(10, activation='softmax')
], name="CustomComponents_Model")

custom_model.compile(optimizer='adam',
                     loss='sparse_categorical_crossentropy',
                     metrics=['accuracy'])

print("Training model with all custom components...")
hist_custom = custom_model.fit(X_train, y_train, validation_data=(X_val, y_val),
                                epochs=15, batch_size=256, verbose=0)

print(f"Best val accuracy: {max(hist_custom.history['val_accuracy']):.4f}")
print("\nComponents used:")
print("  ✓ ParametricSwish activation (learnable beta)")
print("  ✓ VarianceScalingInit initializer")
print("  ✓ SpectralRegularizer (sigma_max penalty)")
print("  ✓ NonNegativeConstraint")


## 8. Custom Metric — Streaming Huber Metric

Keras metrics accumulate values across batches within an epoch and compute the final result at epoch end. We implement a **streaming Huber metric** that maintains running totals without storing every prediction.


In [ ]:
# ============================================================
# 8a — Huber Metric (streaming)
# ============================================================
class HuberMetric(keras.metrics.Metric):
    """
    Streaming Huber metric that accumulates total and count
    across all batches in an epoch.

    Unlike a loss function which computes per-sample values,
    this metric tracks running statistics for epoch-level reporting.

    Args:
        delta: Threshold for switching from quadratic to linear (default: 1.0)
    """
    def __init__(self, delta=1.0, name='huber_metric', **kwargs):
        super().__init__(name=name, **kwargs)
        self.delta = delta
        self.total = self.add_weight(name='total', initializer='zeros')
        self.count = self.add_weight(name='count', initializer='zeros')

    def update_state(self, y_true, y_pred, sample_weight=None):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.cast(y_pred, tf.float32)

        # Flatten for simplicity
        y_true = tf.reshape(y_true, [-1])
        y_pred = tf.reshape(y_pred, [-1])

        error = tf.abs(y_true - y_pred)
        quadratic = tf.minimum(error, self.delta)
        linear = error - quadratic
        huber = 0.5 * quadratic ** 2 + self.delta * linear

        self.total.assign_add(tf.reduce_sum(huber))
        self.count.assign_add(tf.cast(tf.size(y_true), tf.float32))

    def result(self):
        return self.total / tf.maximum(self.count, 1.0)

    def reset_state(self):
        self.total.assign(0.0)
        self.count.assign(0.0)

    def get_config(self):
        return {**super().get_config(), "delta": self.delta}


# ============================================================
# 8b — Also create a Precision-at-K metric
# ============================================================
class TopKAccuracyMetric(keras.metrics.Metric):
    """
    Top-K accuracy: the prediction is correct if the true label
    is among the top-k predicted classes. Useful when classes
    are similar (e.g., shirt vs pullover).
    """
    def __init__(self, k=3, name='top_k_accuracy', **kwargs):
        super().__init__(name=name, **kwargs)
        self.k = k
        self.correct = self.add_weight(name='correct', initializer='zeros')
        self.total = self.add_weight(name='total', initializer='zeros')

    def update_state(self, y_true, y_pred, sample_weight=None):
        y_true = tf.cast(tf.reshape(y_true, [-1]), tf.int32)
        top_k = tf.math.top_k(y_pred, k=self.k).indices
        matches = tf.reduce_any(tf.equal(top_k, y_true[:, tf.newaxis]), axis=1)
        self.correct.assign_add(tf.reduce_sum(tf.cast(matches, tf.float32)))
        self.total.assign_add(tf.cast(tf.size(y_true), tf.float32))

    def result(self):
        return self.correct / tf.maximum(self.total, 1.0)

    def reset_state(self):
        self.correct.assign(0.0)
        self.total.assign(0.0)

# Demo
metric_model = build_simple_cnn()
metric_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy', TopKAccuracyMetric(k=3)]
)

print("Training with custom Top-3 Accuracy metric...")
hist_metric = metric_model.fit(X_train, y_train, validation_data=(X_val, y_val),
                                epochs=10, batch_size=128, verbose=0)

print(f"\nFinal metrics:")
print(f"  Val Accuracy:       {hist_metric.history['val_accuracy'][-1]:.4f}")
print(f"  Val Top-3 Accuracy: {hist_metric.history['val_top_k_accuracy'][-1]:.4f}")


## 9. Custom Layers

Building four different custom layers demonstrating different patterns in the Keras Layer API.


In [ ]:
# ============================================================
# 9a — Exponential Layer (stateless transformation)
# ============================================================
class ExponentialLayer(layers.Layer):
    """
    Computes element-wise exponential: output = exp(input).
    A simple stateless layer — no trainable weights.
    Useful as part of a larger architecture (e.g., log-space models).
    """
    def call(self, inputs):
        return tf.exp(inputs)

# Quick test
exp_layer = ExponentialLayer()
test_input = tf.constant([[0.0, 1.0, 2.0]])
print(f"ExponentialLayer([0, 1, 2]) = {exp_layer(test_input).numpy()}")


In [ ]:
# ============================================================
# 9b — Custom Dense Layer (from scratch)
# ============================================================
class MyDense(layers.Layer):
    """
    Dense layer built entirely from scratch.
    Demonstrates: build(), add_weight(), call(), get_config().

    This is functionally equivalent to keras.layers.Dense but
    written manually to show the full Layer API.
    """
    def __init__(self, units, activation=None, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.activation = keras.activations.get(activation)

    def build(self, input_shape):
        # Create weights dynamically based on input shape
        self.kernel = self.add_weight(
            name='kernel',
            shape=(input_shape[-1], self.units),
            initializer='glorot_uniform',
            trainable=True
        )
        self.bias = self.add_weight(
            name='bias',
            shape=(self.units,),
            initializer='zeros',
            trainable=True
        )
        super().build(input_shape)

    def call(self, inputs):
        z = tf.matmul(inputs, self.kernel) + self.bias
        if self.activation:
            return self.activation(z)
        return z

    def compute_output_shape(self, input_shape):
        return input_shape[:-1] + (self.units,)

    def get_config(self):
        config = super().get_config()
        config.update({
            "units": self.units,
            "activation": keras.activations.serialize(self.activation)
        })
        return config

# Verify it works
test_dense = MyDense(4, activation='relu')
test_out = test_dense(tf.random.normal((2, 8)))
print(f"MyDense(8→4): input shape (2,8) → output shape {test_out.shape}")


In [ ]:
# ============================================================
# 9c — Gaussian Noise Layer (training-only)
# ============================================================
class AddGaussianNoise(layers.Layer):
    """
    Adds Gaussian noise to inputs during training only.
    Acts as a regularizer by making the network robust to
    small perturbations in the input.

    Args:
        stddev: Standard deviation of the noise (default: 0.1)
    """
    def __init__(self, stddev=0.1, **kwargs):
        super().__init__(**kwargs)
        self.stddev = stddev

    def call(self, inputs, training=None):
        if training:
            noise = tf.random.normal(
                shape=tf.shape(inputs),
                mean=0.0,
                stddev=self.stddev,
                dtype=inputs.dtype
            )
            return inputs + noise
        return inputs

    def get_config(self):
        return {**super().get_config(), "stddev": self.stddev}


In [ ]:
# ============================================================
# 9d — Custom Layer Normalization
# ============================================================
class MyLayerNormalization(layers.Layer):
    """
    Layer Normalization: normalizes across the feature dimension
    (not the batch dimension like BatchNorm).

    For each sample independently:
      x_normalized = (x - mean(x)) / (std(x) + eps)
      output = gamma * x_normalized + beta

    Unlike BatchNorm, this works identically at training and inference
    time and doesn't depend on batch statistics.

    Args:
        eps: Small constant for numerical stability (default: 1e-5)
    """
    def __init__(self, eps=1e-5, **kwargs):
        super().__init__(**kwargs)
        self.eps = eps

    def build(self, input_shape):
        feature_shape = input_shape[-1:]
        self.gamma = self.add_weight(
            name='gamma', shape=feature_shape,
            initializer='ones', trainable=True)
        self.beta = self.add_weight(
            name='beta', shape=feature_shape,
            initializer='zeros', trainable=True)
        super().build(input_shape)

    def call(self, inputs):
        mean = tf.reduce_mean(inputs, axis=-1, keepdims=True)
        variance = tf.math.reduce_variance(inputs, axis=-1, keepdims=True)
        normalized = (inputs - mean) / tf.sqrt(variance + self.eps)
        return self.gamma * normalized + self.beta


# ============================================================
# 9e — Model using all custom layers
# ============================================================
custom_layers_model = keras.Sequential([
    layers.Flatten(input_shape=(28, 28, 1)),
    AddGaussianNoise(stddev=0.1),
    MyDense(256, activation='relu'),
    MyLayerNormalization(),
    MyDense(128, activation='relu'),
    MyLayerNormalization(),
    MyDense(10, activation='softmax')
], name="AllCustomLayers")

custom_layers_model.compile(optimizer='adam',
                            loss='sparse_categorical_crossentropy',
                            metrics=['accuracy'])

print("Training model with all custom layers...")
hist_cl = custom_layers_model.fit(X_train, y_train, validation_data=(X_val, y_val),
                                   epochs=15, batch_size=256, verbose=0)

print(f"\nBest val accuracy: {max(hist_cl.history['val_accuracy']):.4f}")
print("Custom layers used: ExponentialLayer, MyDense, AddGaussianNoise, MyLayerNormalization")


## 10. Custom Model — Residual Network

We implement a **ResidualRegressor** with custom `ResidualBlock` modules using the Keras Subclassing API. This demonstrates building complex architectures that can't be expressed as simple Sequential stacks.

**Residual connections** (skip connections) help gradients flow through deep networks by providing a shortcut path, enabling training of much deeper architectures.


In [ ]:
# ============================================================
# 10a — ResidualBlock
# ============================================================
class ResidualBlock(layers.Layer):
    """
    A residual block: output = F(x) + x

    Contains two Dense layers with BatchNorm and optional dropout.
    The skip connection adds the original input to the block output,
    allowing gradients to flow directly through the shortcut.

    If input and output dimensions differ, a projection layer
    is used for the shortcut.
    """
    def __init__(self, units, activation='relu', dropout_rate=0.0, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.dropout_rate = dropout_rate
        self.dense1 = layers.Dense(units, kernel_initializer='he_normal')
        self.bn1 = layers.BatchNormalization()
        self.activation1 = layers.Activation(activation)
        self.dense2 = layers.Dense(units, kernel_initializer='he_normal')
        self.bn2 = layers.BatchNormalization()
        self.activation2 = layers.Activation(activation)
        self.dropout = layers.Dropout(dropout_rate) if dropout_rate > 0 else None
        self.projection = None  # Built lazily

    def build(self, input_shape):
        if input_shape[-1] != self.units:
            self.projection = layers.Dense(self.units, use_bias=False)
        super().build(input_shape)

    def call(self, inputs, training=None):
        # Main path
        z = self.dense1(inputs)
        z = self.bn1(z, training=training)
        z = self.activation1(z)
        if self.dropout:
            z = self.dropout(z, training=training)
        z = self.dense2(z)
        z = self.bn2(z, training=training)

        # Skip connection (with optional projection)
        shortcut = self.projection(inputs) if self.projection else inputs

        return self.activation2(z + shortcut)


# ============================================================
# 10b — ResidualClassifier (full model)
# ============================================================
class ResidualClassifier(keras.Model):
    """
    A deep residual classifier built with the Subclassing API.

    Architecture:
      Input → Flatten → Dense(256) → [ResidualBlock x N] → Dense(num_classes)

    This demonstrates:
    - keras.Model subclassing
    - Composing custom layers
    - Custom call() with training flag propagation
    """
    def __init__(self, num_classes=10, num_blocks=3, block_units=128,
                 dropout_rate=0.2, **kwargs):
        super().__init__(**kwargs)
        self.flatten = layers.Flatten()
        self.input_dense = layers.Dense(256, activation='relu',
                                        kernel_initializer='he_normal')
        self.res_blocks = [
            ResidualBlock(block_units, dropout_rate=dropout_rate,
                         name=f'res_block_{i}')
            for i in range(num_blocks)
        ]
        self.output_dense = layers.Dense(num_classes, activation='softmax')

    def call(self, inputs, training=None):
        x = self.flatten(inputs)
        x = self.input_dense(x)
        for block in self.res_blocks:
            x = block(x, training=training)
        return self.output_dense(x)


# Train
res_model = ResidualClassifier(num_classes=10, num_blocks=4, block_units=128)
res_model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

print("Training Residual Classifier (4 blocks, 128 units)...")
hist_res = res_model.fit(X_train, y_train, validation_data=(X_val, y_val),
                          epochs=20, batch_size=256, verbose=0)

print(f"\nBest val accuracy: {max(hist_res.history['val_accuracy']):.4f}")
res_model.summary()


## 11. Custom Optimizer — Momentum with Nesterov Look-ahead

We implement SGD with momentum from scratch, including the Nesterov improvement which performs a "look-ahead" step before computing the gradient. This gives faster convergence than standard momentum.

**Standard Momentum:** v = β·v - lr·∇L(θ), θ = θ + v

**Nesterov Momentum:** v = β·v - lr·∇L(θ + β·v), θ = θ + v
(Compute gradient at the look-ahead position)


In [ ]:
# ============================================================
# 11a — Custom Momentum Optimizer
# ============================================================
class MyMomentumOptimizer(keras.optimizers.Optimizer):
    """
    Custom SGD with Momentum and optional Nesterov acceleration.

    Implements the velocity-based update rule from scratch:
    1. v_t = beta * v_{t-1} + (1-beta) * gradient
    2. theta = theta - lr * v_t

    With Nesterov: evaluates gradient at the look-ahead position
    for faster convergence on convex problems.

    Args:
        learning_rate: Step size (default: 0.01)
        momentum: Momentum coefficient (default: 0.9)
        nesterov: Whether to use Nesterov acceleration (default: True)
    """
    def __init__(self, learning_rate=0.01, momentum=0.9, nesterov=True,
                 name='MyMomentumOptimizer', **kwargs):
        super().__init__(learning_rate=learning_rate, name=name, **kwargs)
        self.momentum = momentum
        self.nesterov = nesterov

    def build(self, variables):
        super().build(variables)
        # Create velocity buffers for each variable
        self.velocities = []
        for var in variables:
            self.velocities.append(
                self.add_variable_from_reference(var, name='velocity')
            )

    def update_step(self, gradient, variable, learning_rate):
        # Find the velocity for this variable
        idx = self._get_variable_index(variable)
        velocity = self.velocities[idx]

        # Update velocity: v = momentum * v - lr * grad
        new_velocity = self.momentum * velocity - learning_rate * gradient
        velocity.assign(new_velocity)

        if self.nesterov:
            # Nesterov look-ahead: use the projected velocity
            variable.assign_add(self.momentum * new_velocity - learning_rate * gradient)
        else:
            # Standard momentum
            variable.assign_add(new_velocity)

    def _get_variable_index(self, variable):
        for i, v in enumerate(self._variables):
            if v is variable:
                return i
        # Fallback: search by name
        for i, v in enumerate(self._variables):
            if v.name == variable.name:
                return i
        return 0

    def get_config(self):
        config = super().get_config()
        config.update({
            "momentum": self.momentum,
            "nesterov": self.nesterov,
        })
        return config


# A/B test: Adam vs Custom Momentum
print("Training with CUSTOM Momentum Optimizer (Nesterov)...")
mom_model = build_simple_cnn()
mom_model.compile(
    optimizer=MyMomentumOptimizer(learning_rate=0.01, momentum=0.9, nesterov=True),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
hist_mom = mom_model.fit(X_train, y_train, validation_data=(X_val, y_val),
                          epochs=15, batch_size=128, verbose=0)

print(f"Custom Momentum best val acc: {max(hist_mom.history['val_accuracy']):.4f}")
print(f"Adam baseline best val acc:   {max(hist_const.history['val_accuracy']):.4f}")

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(hist_const.history['val_accuracy'], label='Adam')
ax.plot(hist_mom.history['val_accuracy'], label='Custom Nesterov Momentum')
ax.set_title("Optimizer Comparison", fontweight='bold')
ax.set_xlabel("Epoch"); ax.set_ylabel("Val Accuracy")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 12. Custom Training Loop with GradientTape

Instead of `model.fit()`, we write the training loop manually using `tf.GradientTape`. This gives full control over every aspect: gradient computation, accumulation, clipping, logging, and update scheduling.

This is essential for advanced use cases like GANs, reinforcement learning, meta-learning, and mixed-precision training.


In [ ]:
# ============================================================
# 12a — Custom training loop
# ============================================================
# Build model
loop_model = keras.Sequential([
    layers.Flatten(input_shape=(28, 28, 1)),
    layers.Dense(300, activation='relu', kernel_initializer='he_normal'),
    layers.Dense(100, activation='relu', kernel_initializer='he_normal'),
    layers.Dense(10, activation='softmax')
])

optimizer = keras.optimizers.Adam(learning_rate=1e-3)
loss_fn = keras.losses.SparseCategoricalCrossentropy()
train_acc_metric = keras.metrics.SparseCategoricalAccuracy()
val_acc_metric = keras.metrics.SparseCategoricalAccuracy()

# Create tf.data pipeline
train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
train_ds = train_ds.shuffle(10000).batch(128).prefetch(tf.data.AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices((X_val, y_val))
val_ds = val_ds.batch(256).prefetch(tf.data.AUTOTUNE)

# Training step function
@tf.function
def train_step(images, labels):
    """Single training step with GradientTape."""
    with tf.GradientTape() as tape:
        predictions = loop_model(images, training=True)
        loss = loss_fn(labels, predictions)

    # Compute and apply gradients
    gradients = tape.gradient(loss, loop_model.trainable_variables)

    # Gradient clipping for stability
    gradients, global_norm = tf.clip_by_global_norm(gradients, 5.0)

    optimizer.apply_gradients(zip(gradients, loop_model.trainable_variables))
    train_acc_metric.update_state(labels, predictions)
    return loss, global_norm


@tf.function
def val_step(images, labels):
    """Validation step (no gradient computation)."""
    predictions = loop_model(images, training=False)
    loss = loss_fn(labels, predictions)
    val_acc_metric.update_state(labels, predictions)
    return loss


# Main training loop
EPOCHS = 15
history = {'train_loss': [], 'val_loss': [],
           'train_acc': [], 'val_acc': [],
           'grad_norms': []}

print("Custom Training Loop — Fashion MNIST")
print("=" * 60)

for epoch in range(EPOCHS):
    # --- Training ---
    train_losses = []
    grad_norms_epoch = []

    for batch_images, batch_labels in train_ds:
        loss, grad_norm = train_step(batch_images, batch_labels)
        train_losses.append(loss.numpy())
        grad_norms_epoch.append(grad_norm.numpy())

    train_loss = np.mean(train_losses)
    train_acc = train_acc_metric.result().numpy()
    train_acc_metric.reset_state()

    # --- Validation ---
    val_losses = []
    for batch_images, batch_labels in val_ds:
        loss = val_step(batch_images, batch_labels)
        val_losses.append(loss.numpy())

    val_loss = np.mean(val_losses)
    val_acc = val_acc_metric.result().numpy()
    val_acc_metric.reset_state()

    # Log
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    history['grad_norms'].append(np.mean(grad_norms_epoch))

    print(f"  Epoch {epoch+1:2d}/{EPOCHS} | "
          f"Loss: {train_loss:.4f} / {val_loss:.4f} | "
          f"Acc: {train_acc:.4f} / {val_acc:.4f} | "
          f"Grad Norm: {np.mean(grad_norms_epoch):.4f}")


In [ ]:
# ============================================================
# 12b — Visualize custom training loop results
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

axes[0].plot(history['train_loss'], label='Train')
axes[0].plot(history['val_loss'], label='Val')
axes[0].set_title("Loss", fontweight='bold')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(history['train_acc'], label='Train')
axes[1].plot(history['val_acc'], label='Val')
axes[1].set_title("Accuracy", fontweight='bold')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

axes[2].plot(history['grad_norms'], color='orange')
axes[2].set_title("Average Gradient Norm", fontweight='bold')
axes[2].grid(True, alpha=0.3)

plt.suptitle("Custom Training Loop Results", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Final test eval
test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(256)
test_metric = keras.metrics.SparseCategoricalAccuracy()
for images, labels in test_ds:
    preds = loop_model(images, training=False)
    test_metric.update_state(labels, preds)
print(f"\nFinal Test Accuracy: {test_metric.result().numpy():.4f}")


## 13. Weights & Biases (W&B) Integration

[Weights & Biases](https://wandb.ai/) is a platform for experiment tracking, model versioning, and collaboration. It provides interactive dashboards, hyperparameter sweeps, and artifact management.

We demonstrate:
1. Logging metrics during training
2. Logging hyperparameters
3. Logging model architecture
4. Logging sample predictions as images


In [ ]:
# ============================================================
# 13a — W&B Setup and Training
# ============================================================
import wandb
from wandb.integration.keras import WandbMetricsLogger, WandbModelCheckpoint

# Initialize W&B run (anonymous mode for demo)
# In production, use: wandb.login(key="YOUR_API_KEY")
wandb.init(
    project="advanced-tf-constructs",
    name="fashion-mnist-demo",
    config={
        "architecture": "CNN",
        "dataset": "Fashion-MNIST",
        "epochs": 15,
        "batch_size": 128,
        "learning_rate": 1e-3,
        "optimizer": "Adam",
        "dropout": 0.3,
        "conv_filters": [32, 64],
        "dense_units": 128,
    },
    mode="offline"  # Use "online" with API key for cloud logging
)

config = wandb.config
print(f"W&B run initialized: {wandb.run.name}")
print(f"Config: {dict(config)}")


In [ ]:
# ============================================================
# 13b — Train with W&B logging
# ============================================================
# Build model
wandb_model = keras.Sequential([
    layers.Conv2D(config.conv_filters[0], 3, padding='same',
                  activation='relu', input_shape=(28, 28, 1)),
    layers.MaxPooling2D(2),
    layers.Conv2D(config.conv_filters[1], 3, padding='same', activation='relu'),
    layers.MaxPooling2D(2),
    layers.Flatten(),
    layers.Dense(config.dense_units, activation='relu'),
    layers.Dropout(config.dropout),
    layers.Dense(10, activation='softmax')
])

wandb_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=config.learning_rate),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Custom W&B callback for extra logging
class WandBDetailedLogger(keras.callbacks.Callback):
    """Log additional metrics to W&B: confusion matrix, sample predictions."""
    def on_epoch_end(self, epoch, logs=None):
        # Log standard metrics
        wandb.log({
            "epoch": epoch,
            "train_loss": logs.get('loss'),
            "train_accuracy": logs.get('accuracy'),
            "val_loss": logs.get('val_loss'),
            "val_accuracy": logs.get('val_accuracy'),
        })

        # Log sample predictions every 5 epochs
        if epoch % 5 == 0:
            preds = self.model.predict(X_val[:32], verbose=0)
            pred_classes = np.argmax(preds, axis=1)
            true_classes = y_val[:32]

            # Log as W&B table
            table = wandb.Table(columns=["Image", "Predicted", "True", "Correct"])
            for i in range(min(16, len(pred_classes))):
                img = wandb.Image(X_val[i, :, :, 0])
                table.add_data(
                    img,
                    CLASS_NAMES[pred_classes[i]],
                    CLASS_NAMES[true_classes[i]],
                    pred_classes[i] == true_classes[i]
                )
            wandb.log({f"predictions_epoch_{epoch}": table})


# Train
print("Training with W&B logging...")
wandb_history = wandb_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=config.epochs,
    batch_size=config.batch_size,
    verbose=0,
    callbacks=[WandBDetailedLogger()]
)

# Log final test results
test_loss, test_acc = wandb_model.evaluate(X_test, y_test, verbose=0)
wandb.log({"test_loss": test_loss, "test_accuracy": test_acc})

# Log model summary
wandb.log({"model_parameters": wandb_model.count_params()})

print(f"\nFinal Test Accuracy: {test_acc:.4f}")
print(f"Total Parameters: {wandb_model.count_params():,}")

# Finish the run
wandb.finish()
print("\nW&B run finished. View results at: https://wandb.ai/")
print("For online mode, set mode='online' and login with wandb.login()")


## Summary — All Custom Components

| # | Component | Class/Function | Key Concept |
|---|-----------|---------------|-------------|
| 1 | **LR Scheduler** | `OneCycleScheduler` | Cosine warmup → annealing for fast convergence |
| 2 | **Custom Dropout** | `MCAlphaDropout` | SELU-compatible dropout with MC inference |
| 3 | **Custom Normalization** | `MaxNormDense` | Per-neuron weight norm capping |
| 4 | **TensorBoard** | `DetailedTBCallback` | Gradients, weight stats, prediction images |
| 5 | **Custom Loss** | `HuberLoss`, `QuantileLoss` | Outlier-robust regression |
| 6 | **Custom Activation** | `ParametricSwish` | Learnable Swish with channel-wise beta |
| 6 | **Custom Initializer** | `VarianceScalingInit` | Configurable fan-based scaling |
| 6 | **Custom Regularizer** | `SpectralRegularizer` | Bounds largest singular value |
| 6 | **Custom Constraint** | `NonNegativeConstraint` | Forces weights ≥ 0 |
| 7 | **Custom Metric** | `TopKAccuracyMetric` | Streaming top-k accuracy |
| 8 | **Custom Layers** | `MyDense`, `AddGaussianNoise`, `MyLayerNormalization` | Full Layer API |
| 9 | **Custom Model** | `ResidualClassifier` | Subclassing API with skip connections |
| 10 | **Custom Optimizer** | `MyMomentumOptimizer` | SGD + Nesterov from scratch |
| 11 | **Custom Training Loop** | `GradientTape` | Full control, gradient clipping |
| 12 | **W&B Integration** | `wandb` | Experiment tracking, tables, artifacts |

---
*Notebook generated for Assignment Part 2A — TensorFlow Edition*
